# API-Football (RapidAPI) Ingestion

Pull NFL player statistics from API-Football via RapidAPI and land in bronze/silver Delta tables.

**Features:**
- Free tier: 100 requests/day
- NFL endpoints and statistics
- Game results, standings, and player stats
- Real-time data during games

**Resources:**
- Website: https://rapidapi.com/api-sports/api/api-football
- Docs: https://api-sports.io/documentation/nfl/v1
- Sign up: https://rapidapi.com/auth/sign-up

**Setup:**
1. Create account at RapidAPI
2. Subscribe to API-Football (free tier)
3. Get your API key from dashboard
4. Set API key in Databricks Secrets: `dbutils.secrets.put(scope="api-keys", key="rapidapi-key", string_value="YOUR_KEY")`

In [0]:
import requests
import pandas as pd
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BASE_URL = "https://api-football-v1.p.rapidapi.com/v3"
SEASON = 2024
WEEK = 18

# Get API key from Databricks Secrets
# Create secret scope first: databricks secrets create-scope --scope api-keys
# Add key: databricks secrets put --scope api-keys --key rapidapi-key
try:
    API_KEY = dbutils.secrets.get(scope="api-keys", key="rapidapi-key")
    print("✓ API key loaded from secrets")
except:
    print("❌ ERROR: API key not found in secrets")
    print("\nTo set up:")
    print("1. Create secret scope: databricks secrets create-scope --scope api-keys")
    print("2. Add key: databricks secrets put --scope api-keys --key rapidapi-key")
    print("3. Or set manually: API_KEY = 'your-key-here'")
    # Uncomment and set your key manually for testing
    # API_KEY = "your-rapidapi-key-here"
    raise

print(f"\n📅 Fetching API-Football data for Week {WEEK}, Season {SEASON}")
print(f"API Endpoint: {BASE_URL}")
print(f"Rate Limit: 100 requests/day (free tier)")

In [0]:
# First, get games for the week to extract player stats
print("Fetching NFL games for the week...")

headers = {
    "X-RapidAPI-Key": API_KEY,
    "X-RapidAPI-Host": "api-football-v1.p.rapidapi.com"
}

try:
    # Get games for the season/week
    # Note: API-Football uses league ID 1 for NFL
    url = f"{BASE_URL}/games"
    params = {
        "league": "1",  # NFL league ID
        "season": str(SEASON)
    }
    
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    
    if 'response' in data:
        games = data['response']
        print(f"✓ Fetched {len(games)} games")
        
        # Filter for specific week if needed
        # API-Football may organize by date rather than week number
        print(f"\nSample game data:")
        if len(games) > 0:
            print(json.dumps(games[0], indent=2))
    else:
        print("❌ No games found in response")
        print(f"Response: {data}")
        
except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching games: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"Response: {e.response.text}")
    raise

In [0]:
# Fetch player statistics
# Note: API-Football free tier has limited endpoints
# You may need to aggregate from game statistics

print("\nFetching player statistics...")

try:
    # Get player statistics endpoint
    # This varies by API-Football's NFL coverage
    url = f"{BASE_URL}/players/statistics"
    params = {
        "league": "1",
        "season": str(SEASON)
    }
    
    response = requests.get(url, headers=headers, params=params, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        
        if 'response' in data and len(data['response']) > 0:
            players = data['response']
            print(f"✓ Fetched {len(players)} player records")
            
            # Convert to pandas DataFrame
            players_df = pd.DataFrame(players)
            
            print(f"\nAvailable columns: {list(players_df.columns)}")
            print(f"\nSample data:")
            display(players_df.head(5))
        else:
            print("❌ No player data available")
            print(f"Response: {data}")
            # Create empty DataFrame as fallback
            players_df = pd.DataFrame()
    else:
        print(f"❌ HTTP {response.status_code}: {response.text}")
        # For free tier limitations, you may need to use alternative endpoints
        print("\n💡 Tip: Free tier may have limited player statistics access")
        print("Consider using game statistics endpoint and aggregating manually")
        players_df = pd.DataFrame()
        
except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching player stats: {e}")
    players_df = pd.DataFrame()

In [0]:
# Transform to Spark DataFrame (if data is available)
if 'players_df' in locals() and len(players_df) > 0:
    print("Transforming to Spark DataFrame...")
    
    rows = []
    for idx, row in players_df.iterrows():
        # Map API-Football fields to standard schema
        # Field names may vary - adjust based on actual response
        player_data = row.get('player', {}) if isinstance(row.get('player'), dict) else {}
        stats = row.get('statistics', [{}])[0] if isinstance(row.get('statistics'), list) else {}
        
        player_id = str(player_data.get('id', f'unknown_{idx}'))
        player_name = player_data.get('name', '')
        position = stats.get('games', {}).get('position', '')
        team = stats.get('team', {}).get('name', '')
        
        # Extract fantasy-relevant stats
        # Adjust field names based on actual API response
        passing_yards = stats.get('passing', {}).get('yards', 0)
        passing_tds = stats.get('passing', {}).get('touchdowns', 0)
        rushing_yards = stats.get('rushing', {}).get('yards', 0)
        rushing_tds = stats.get('rushing', {}).get('touchdowns', 0)
        receptions = stats.get('receiving', {}).get('receptions', 0)
        receiving_yards = stats.get('receiving', {}).get('yards', 0)
        receiving_tds = stats.get('receiving', {}).get('touchdowns', 0)
        
        # Calculate PPR fantasy points
        fantasy_points = (
            (passing_yards * 0.04) +
            (passing_tds * 4) +
            (rushing_yards * 0.1) +
            (rushing_tds * 6) +
            (receptions * 1) +
            (receiving_yards * 0.1) +
            (receiving_tds * 6)
        )
        
        stats_json = json.dumps(row.to_dict(), default=str)
        
        rows.append(
            Row(
                player_id=player_id,
                player_name=player_name,
                position=position,
                team=team,
                week=WEEK,
                season=SEASON,
                fantasy_points=fantasy_points,
                stats=stats_json
            )
        )
    
    stats_df = spark.createDataFrame(rows)
    print(f"✓ Created Spark DataFrame with {stats_df.count()} records")
    display(stats_df.limit(10))
    
else:
    print("⚠️ No player data available to transform")
    print("\nNote: API-Football free tier may have limited player statistics.")
    print("Consider upgrading to paid tier or using alternative data sources.")

In [0]:
# Write to bronze table (if data available)
if 'stats_df' in locals() and stats_df.count() > 0:
    print("Writing to bronze_weekly_stats...")
    
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    bronze_df = bronze_df.withColumn("source", F.lit("api_football_rapidapi"))
    
    bronze_df.createOrReplaceTempView("api_football_bronze_updates")
    
    spark.sql("""
      MERGE INTO main.fantasai.bronze_weekly_stats AS target
      USING api_football_bronze_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.player_name = source.player_name,
          target.position = source.position,
          target.team = source.team,
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.source = source.source,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, player_name, position, team, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.player_id, source.player_name, source.position, source.team, source.week, source.season,
                source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {bronze_df.count()} records from API-Football into bronze_weekly_stats")
    
else:
    print("⚠️ Skipping bronze write - no data available")

In [0]:
# Transform for silver table (if data available)
if 'bronze_df' in locals():
    print("Transforming and writing to silver_weekly_stats...")
    
    silver_df = (
        bronze_df
        .select(
            F.col("player_id").cast("string"),
            F.col("player_name").cast("string"),
            F.col("position").cast("string"),
            F.col("team").cast("string"),
            F.col("week").cast("int"),
            F.col("season").cast("int"),
            F.col("fantasy_points").cast("double"),
            F.col("stats").cast("string"),
            F.col("source"),
            F.col("ingested_at"),
        )
        .dropDuplicates(["player_id", "week", "season"])
        .filter(F.col("fantasy_points") > 0)
    )
    
    silver_df.createOrReplaceTempView("api_football_silver_updates")
    
    spark.sql("""
      MERGE INTO main.fantasai.silver_weekly_stats AS target
      USING api_football_silver_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.player_name = source.player_name,
          target.position = source.position,
          target.team = source.team,
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.source = source.source,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, player_name, position, team, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.player_id, source.player_name, source.position, source.team, source.week, source.season,
                source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {silver_df.count()} records from API-Football into silver_weekly_stats")
    
else:
    print("⚠️ Skipping silver write - no data available")

In [0]:
%sql
-- Check API-Football data in silver table
SELECT 
  player_id,
  player_name,
  position,
  team,
  week,
  season,
  fantasy_points,
  source
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2024 AND source = 'api_football_rapidapi'
ORDER BY fantasy_points DESC
LIMIT 20

## API-Football (RapidAPI) Features

### Available Endpoints
1. **Games** - `/games` - Game schedules and results
2. **Standings** - `/standings` - Team standings
3. **Players** - `/players` - Player information
4. **Statistics** - `/players/statistics` - Player statistics (limited on free tier)

### Pricing Tiers
- **Free**: 100 requests/day
- **Basic**: $9.99/month - 1,000 requests/day
- **Pro**: $29.99/month - 10,000 requests/day
- **Ultra**: $79.99/month - 100,000 requests/day

### Key Advantages
- ✅ Real-time data during games
- ✅ NFL coverage with game results
- ✅ Free tier available for testing

### Limitations
- ❌ **Free tier very limited** (100 req/day)
- ❌ Player statistics may require paid tier
- ❌ May need to aggregate from game-level data

### Recommendation
**For fantasy football use cases, consider:**
- **nflverse** (free, unlimited) - Better for weekly stats
- **Fantasy Football Data Pros** (free, unlimited) - Better for player stats
- **API-Football** - Good for real-time game data if you have paid tier

### Rate Limit Management
```python
# Check your rate limit status
headers = {"X-RapidAPI-Key": API_KEY, "X-RapidAPI-Host": "api-football-v1.p.rapidapi.com"}
response = requests.get(f"{BASE_URL}/status", headers=headers)
print(response.json())
```